# 第14・15回 動的計画法の実験
音声なしの補助ノートブックです。状態、境界、遷移、計算順序、復元を確認します。上から実行してください。

In [ ]:
import matplotlib.pyplot as plt
from itertools import product

## ナップサック：D(i,b)
先頭i個を使い、予算b以下で得られる最大満足度。各品物は一度だけ選べます。

In [ ]:
prices=[60, 30, 40, 50, 40, 30]
values=[36, 27, 12, 50, 28, 24]
B=100
D=[[0]*(B+1) for _ in range(len(prices)+1)]
for i,(p,u) in enumerate(zip(prices,values),1):
    for b in range(B+1):
        D[i][b]=D[i-1][b]
        if b>=p: D[i][b]=max(D[i][b],D[i-1][b-p]+u)
print("最適値:",D[-1][B])
b=B; chosen=[]
for i in range(len(prices),0,-1):
    if D[i][b] != D[i-1][b]:
        chosen.append(i); b-=prices[i-1]
print("選択:",sorted(chosen))
assert D[-1][B]==79
plt.imshow(D,aspect="auto",origin="lower");plt.xlabel("budget");plt.ylabel("items");plt.colorbar();plt.show()

## 編集距離
挿入・削除・置換の費用1、一致0。空文字を含め、表を1行1列大きくします。

In [ ]:
def edit_distance(A,B):
    D=[[0]*(len(B)+1) for _ in range(len(A)+1)]
    P={}
    for i in range(1,len(A)+1): D[i][0]=i;P[i,0]=(i-1,0,"delete")
    for j in range(1,len(B)+1): D[0][j]=j;P[0,j]=(0,j-1,"insert")
    for i in range(1,len(A)+1):
        for j in range(1,len(B)+1):
            delta=int(A[i-1]!=B[j-1])
            options=[(D[i-1][j-1]+delta,(i-1,j-1,"replace" if delta else "match")),(D[i-1][j]+1,(i-1,j,"delete")),(D[i][j-1]+1,(i,j-1,"insert"))]
            D[i][j],P[i,j]=min(options,key=lambda x:x[0])
    i,j=len(A),len(B);ops=[]
    while i or j:
        pi,pj,op=P[i,j];ops.append((i,j,op));i,j=pi,pj
    return D,list(reversed(ops))
D,ops=edit_distance("cart","cat")
print(*D,sep="\n");print(ops);assert D[-1][-1]==1

### 練習
AGCをATへ変える最小回数を予想し、表と操作列を確認してください。

In [ ]:
D,ops=edit_distance("AGC","AT")
print(D[-1][-1],ops)
assert D[-1][-1]==2

## DNAの大域アラインメント
一致+1、A↔GまたはC↔Tは+0.2、他の置換0、ギャップ−1という授業用の規則です。整数10倍で計算し、小数誤差を避けます。実用で唯一の採点規則という意味ではありません。

In [ ]:
def align(A,B):
    def score(a,b):
        return 10 if a==b else 2 if {a,b} in ({"A","G"},{"C","T"}) else 0
    S=[[0]*(len(B)+1) for _ in range(len(A)+1)];P={}
    for i in range(1,len(A)+1): S[i][0]=-10*i;P[i,0]=(i-1,0)
    for j in range(1,len(B)+1): S[0][j]=-10*j;P[0,j]=(0,j-1)
    for i in range(1,len(A)+1):
        for j in range(1,len(B)+1):
            S[i][j],P[i,j]=max([(S[i-1][j-1]+score(A[i-1],B[j-1]),(i-1,j-1)),(S[i-1][j]-10,(i-1,j)),(S[i][j-1]-10,(i,j-1))],key=lambda z:z[0])
    i,j=len(A),len(B);aa=[];bb=[]
    while i or j:
        pi,pj=P[i,j];aa.append(A[i-1] if pi<i else "-");bb.append(B[j-1] if pj<j else "-");i,j=pi,pj
    return S[-1][-1]/10,"".join(reversed(aa)),"".join(reversed(bb)),S
for A,B in [("AGC","AT"),("ACA","GA")]:
    score,aa,bb,S=align(A,B);print(score,aa,bb,sep="\n");assert score==0.2
    print("DP表（行は空文字＋" + A + "、列は空文字＋" + B + "）")
    for row in S:
        print([value / 10 for value in row])


## ふり返り
編集距離は最小、類似度スコアは最大です。配列全体を対応付ける大域アラインメントと、類似部分を探す局所アラインメントを区別します。BLASTは局所的な類似を高速検索する仕組みで、今回の単純な大域DPと同一ではありません。